# Previsão de Intenção de Compra de Clientes em Loja Web

**Descrição do Projeto**

Neste projeto, o objetivo é criar um sistema inteligente para antecipar a intenção de compra dos clientes em um site de e-commerce. A intenção é prever quais clientes têm maior probabilidade de realizar compras online, com base em suas características e comportamentos passados. Essa capacidade de prever a intenção de compra não só aprimora a experiência do cliente, como também permite que a empresa direcione seus esforços de marketing de forma mais eficaz.

**Objetivo**

Desenvolver um modelo preditivo capaz de analisar os padrões de comportamento dos clientes e identificar sinais que indicam a propensão deles para realizar compras no site da empresa. Para isso, será usada uma base de dados que contém informações detalhadas sobre os clientes, incluindo:
- Dados demográficos (idade, renda, etc.)
- Informações sobre compras anteriores

**Base de dados**:
- Year_Birth: Ano de nascimento do cliente.
- Education: Nível de escolaridade do cliente.
- Marital_Status: Estado civil do cliente.
- Income: Renda anual da família do cliente.
- Kidhome: Número de crianças na casa do cliente.
- Recency: Número de dias desde a última compra do cliente.
- Complain: 1 se o cliente reclamou nos últimos 2 anos, 0 caso contrário.
- MntWines: Valor gasto em vinhos nos últimos 2 anos.
- MntFruits: Valor gasto em frutas nos últimos 2 anos.
- MntMeatProducts: Valor gasto em carnes nos últimos 2 anos.
- MntFishProducts: Valor gasto em peixes nos últimos 2 anos.
- MntSweetProducts: Valor gasto em doces nos últimos 2 anos.
- MntGoldProds: Valor gasto em produtos de ouro nos últimos 2 anos.
- NumDealsPurchases: Número de compras feitas com desconto
- NumStorePurchases: Número de compras feitas diretamente nas lojas.
- NumWebVisitsMonth: Número de visitas ao site da empresa no último mês.
- Variável alvo:
    - **WebPurchases: Número de compras feitas pelo site da empresa.**

# Preparação dos Dados

## **Exploração e Limpeza:**

Analise e limpeza dos dados para a garantia de que estejam prontos para a modelagem.


**Configurações**

In [282]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
# import matplotlib.pyplot as plt
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.metrics import classification_report, confusion_matrix

In [393]:
df = pd.read_csv('marketing_campaign.csv', delimiter=';')
df.head(3)

,Year_Birth,Education,Marital_Status,Income,Kidhome,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,NumStorePurchases,NumWebVisitsMonth,Complain,WebPurchases
0,1957,Graduation,Single,58138.0,0,58,635,88,546,172,88,88,4,7,0,1
1,1954,Graduation,Single,46344.0,1,38,11,1,6,2,1,6,2,5,0,0
2,1965,Graduation,Together,71613.0,0,26,426,49,127,111,21,42,10,4,0,1


**Correções de texto**

In [394]:
df.dtypes

Year_Birth             int64
Education             object
Marital_Status        object
Income               float64
Kidhome                int64
Recency                int64
MntWines               int64
MntFruits              int64
MntMeatProducts        int64
MntFishProducts        int64
MntSweetProducts       int64
MntGoldProds           int64
NumStorePurchases      int64
NumWebVisitsMonth      int64
Complain               int64
WebPurchases           int64
dtype: object

In [395]:
for i in df:
    if df[i].dtype == 'object':
        print(df[i].unique())

['Graduation' 'PhD' 'Master' 'Basic' '2n Cycle']
['Single' 'Together' 'Married' 'Divorced' 'Widow' 'Alone' 'Absurd' 'YOLO']


O campo "Education" não apresenta nenhum valor desconhecido ou errado.  
O campo "Martial Status" apresenta dois valores desconhecidos, "Absurd" e "YOLO", além de alguns status incomuns como "Together", "Widow" e "Alone". Vamos ver o que eles representam na base:

In [396]:
df['Marital_Status'].value_counts()

Marital_Status
Married     864
Together    580
Single      480
Divorced    232
Widow        77
Alone         3
Absurd        2
YOLO          2
Name: count, dtype: int64

In [397]:
print(f'O percentual que as categorias Absurd, YOLO e Alone representam é: {100*df['Marital_Status'].isin(['Absurd', 'YOLO', 'Alone']).sum()/df.index.size:.3f}%')
print(f'O percentual que a categoria Widow representa é: {100*df['Marital_Status'].isin(['Widow']).sum()/df.index.size:.3f}%')
print(f'O percentual que a categoria Together representa é: {100*df['Marital_Status'].isin(['Together']).sum()/df.index.size:.3f}%')

O percentual que as categorias Absurd, YOLO e Alone representam é: 0.312%
O percentual que a categoria Widow representa é: 3.438%
O percentual que a categoria Together representa é: 25.893%


Como as categorias Absurd, YOLO e Alone não representam uma quantidade razoável, optei por corrigí-las para "Single".

A categoria Togeher represente uma parcela significativa da amostra, indicando que uma grande quantidade de clientes se identificam com essa categoria, e não com Married. Isso pode significar um comportamento divergente entre as duas categorias.

Já o valor "Widow" representa uma amostra muito pequena da amostra. Manter a categoria pode acrescentar "ruído" a análise, enquanto a substituir pode omitir um comportamento de nicho. Optei por manter a categoria avaliar seu impacto ao longo da análise. 

In [398]:
for campo in ['Absurd', 'YOLO', 'Alone']:
     df.replace({'Marital_Status': campo}, 'Single', inplace=True)
df['Marital_Status'].value_counts()

Marital_Status
Married     864
Together    580
Single      487
Divorced    232
Widow        77
Name: count, dtype: int64

### **Inspeção de possívei outliers**

O objetivo é avaliar se existem possíveis outlieres para guiar as decisões seguitens, ainda que eles possam não ser resolvidos nessa etapa.

In [399]:
df.describe()

,Year_Birth,Income,Kidhome,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,NumStorePurchases,NumWebVisitsMonth,Complain,WebPurchases
count,2240.000000,2216.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000
mean,1968.805804,52247.251354,0.444196,49.109375,303.935714,26.302232,166.950000,37.525446,27.062946,44.021875,5.790179,5.316518,0.009375,0.503571
std,11.984069,25173.076661,0.538398,28.962453,336.597393,39.773434,225.715373,54.628979,41.280498,52.167439,3.250958,2.426645,0.096391,0.500099
min,1893.000000,1730.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1959.000000,35303.000000,0.000000,24.000000,23.750000,1.000000,16.000000,3.000000,1.000000,9.000000,3.000000,3.000000,0.000000,0.000000
50%,1970.000000,51381.500000,0.000000,49.000000,173.500000,8.000000,67.000000,12.000000,8.000000,24.000000,5.000000,6.000000,0.000000,1.000000
75%,1977.000000,68522.000000,1.000000,74.000000,504.250000,33.000000,232.000000,50.000000,33.000000,56.000000,8.000000,7.000000,0.000000,1.000000
max,1996.000000,666666.000000,2.000000,99.000000,1493.000000,199.000000,1725.000000,259.000000,263.000000,362.000000,13.000000,20.000000,1.000000,1.000000


O valor máximo de income está alto,  O valor máximo de Income é muito maior do que 75%, mesmo que a variância seja somada a ele. Esse salto no valor deve ser investigado.

O valor max de todas as compras se destacam, mas ainda estão dentro do esperado para o intervalo de 2 anos. Os valores são razoáveis para um cliente frequente que compra produtos a dois anos na loja analisada, então não apresentam comportamento de outlier.

Os demais valores apresentam comportamento dentro do esperado.

Na próxima etapa discutimos a presença de valores nulos em Income. Antes de corrigir esses números com os valores médios dessa variável, é preciso corrigir possíveis outliers.

In [400]:
campo = 'Income'
fig = make_subplots(rows=1, cols=2, subplot_titles=[f'Histograma de {campo}', f'Box Plot de {campo}'])
fig.add_trace(
    px.histogram(df, x=campo, histnorm="percent", nbins=60).data[0],
    row=1, col=1
    )
fig.add_trace(
    px.box(df, y=campo).data[0],
    row=1, col=2
    )
fig.update_layout(title_text=f'{campo}', showlegend=False)
fig.show()

A partir dos gráficos acima fica evidente que o atual valor máximo contém algum erro. Para simplificar e evitar a indução ao erro, o valor será excluído. Os demais valores que estão fora do boxplot não serão excluídos pois seu comportamento está de acordo com um comportamento padrão da distribuição desigual de renda. Segue abaixo a distribuição após a exclusão do valor máximo.

In [401]:
df = df[df['Income']!=df['Income'].max()].reset_index()
df.drop('index', axis=1, inplace=True)

In [403]:
campo = 'Income'
fig = make_subplots(rows=1, cols=2, subplot_titles=[f'Histograma de {campo}', f'Box Plot de {campo}'])
fig.add_trace(
    px.histogram(df, x=campo, histnorm="percent", nbins=60).data[0],
    row=1, col=1
    )
fig.add_trace(
    px.box(df, y=campo).data[0],
    row=1, col=2
    )
fig.update_layout(title_text=f'{campo}', showlegend=False)
fig.show()

### Limpeza de valores nulos e faltantes

In [404]:
df.count()

Year_Birth           2239
Education            2239
Marital_Status       2239
Income               2215
Kidhome              2239
Recency              2239
MntWines             2239
MntFruits            2239
MntMeatProducts      2239
MntFishProducts      2239
MntSweetProducts     2239
MntGoldProds         2239
NumStorePurchases    2239
NumWebVisitsMonth    2239
Complain             2239
WebPurchases         2239
dtype: int64

In [405]:
df.isnull().sum()

Year_Birth            0
Education             0
Marital_Status        0
Income               24
Kidhome               0
Recency               0
MntWines              0
MntFruits             0
MntMeatProducts       0
MntFishProducts       0
MntSweetProducts      0
MntGoldProds          0
NumStorePurchases     0
NumWebVisitsMonth     0
Complain              0
WebPurchases          0
dtype: int64

In [406]:
df.sort_values('Income').tail(24)

,Year_Birth,Education,Marital_Status,Income,Kidhome,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,NumStorePurchases,NumWebVisitsMonth,Complain,WebPurchases
10,1983,Graduation,Married,NaN,1,11,5,5,6,0,2,1,2,7,0,0
27,1986,Graduation,Single,NaN,1,19,5,1,3,3,263,362,0,1,0,1
43,1959,PhD,Single,NaN,0,80,81,11,50,3,2,39,4,2,0,0
48,1951,Graduation,Single,NaN,2,96,48,5,48,6,10,7,4,6,0,0
58,1982,Graduation,Single,NaN,1,57,11,3,22,2,2,6,3,6,0,0
71,1973,2n Cycle,Married,NaN,1,25,25,3,43,17,4,17,3,8,0,0
90,1957,PhD,Married,NaN,2,4,230,42,192,49,37,53,8,9,0,1
91,1957,Graduation,Single,NaN,1,45,7,0,8,2,0,1,2,7,0,0
92,1973,Master,Together,NaN,0,87,445,37,359,98,28,18,8,1,0,0
128,1961,PhD,Married,NaN,0,23,352,0,27,10,0,15,7,6,0,1


Percentual que os nulos representam:

In [407]:
print(f'{100*df['Income'].isnull().sum()/df.index.size:.3f}%')

1.072%


Os valores nulos representam menos de 2% do total de dados, então podem ser excluídos sem grande perda. Contudo, com o objetivo de me desafiar e não perder dados, optei por substituí-los por outro valor. Ao invés de substituí-los pela média simples, eles foram substituídos pela média de Income para cada conjunto com o mesmo marital status e education.


O primeiro passo é registrar o valor dos índeces referentes aos valores nulos.

In [408]:
# Registro do index de cada valor nulo

index_geral=0
index_null = []
for i in df['Income'].isnull() :
    if i == True:
        index_null.append(index_geral)
    index_geral += 1
print(f'Os índices estão corretos se todos os valores de Income para os respectivos forem nulos e seu tamanho for 24:\nTamanho: {len(index_null)}\nSão nulos: {df['Income'].iloc[index_null].isnull().sum()}')

Os índices estão corretos se todos os valores de Income para os respectivos forem nulos e seu tamanho for 24:
Tamanho: 24
São nulos: 24


Para calcular a média para cada grupo, basta calcular a média agrupada. Dessa forma a variável irá guardar os valores médios de Income para cada par de valores em martial status e education.

In [409]:
# Cálculo da média por grupo
media_income_MS_Ed = df.groupby(['Marital_Status', 'Education'])['Income'].mean()
print(f'Por exemplo, para marital status = together e education = master, a é dia é: {media_income_MS_Ed['Together','Master']}')

Por exemplo, para marital status = together e education = master, a é dia é: 52109.009803921566


Agora, para cada índice registrado em index_null, o valor Nan de Income será substituído pela respectiva média para o mesmo par de Martial Status e Education.

In [410]:
# Substituição pela respectiva média
for i in index_null:
    # df['Income'].iloc[i] = media_income_MS_Ed[df['Marital_Status'].iloc[i], df['Education'].iloc[i]] 
    # df.loc[row_indexer, "col"] = values
    df.loc[i, 'Income'] = media_income_MS_Ed[df.loc[i, 'Marital_Status'], df.loc[i, 'Education']]
print(f'A substiuição foi bem sucedida se a soma de valores nulos for zero: {df['Income'].isnull().sum()}')

A substiuição foi bem sucedida se a soma de valores nulos for zero: 0



## **Análise:** Construção do storytelling com gráficos, analisando e retirando insights das informações.

# ETAPA 2:
**Pré-processamento**

**Análise Correlação:** Verifique a correlação entre as váriaveis e análise se há espaço para retirar váriaveis que não te parecem importantes.

**Codificação de Variáveis Categóricas:** Transformar variáveis categóricas em um formato que os modelos de machine learning possam interpretar.


**Separe a base em Y, X e Treino e teste:**: Faça a separação da base.

**Realize a padronização dos dados**: Padronize os dados para garantir eficiência no modelo e eficácia.








In [ ]:
# seu código aqui

# ETAPA 3:

**Modelagem**

Escolha ao menos 2 técnicas de machine learning e rode 2 modelos, afim de identificar qual tem o melhor resultado para essa base. Lembrando que estamos lidando com uma classificação binária.

In [ ]:
# seu código aqui

# ETAPA 4:

**Avaliação**

Avalie os resultados encontrados nos dois modelos e identifique qual te pareceu realizar melhor as previsões.

Utilize além das métricas padrões a matriz de confusão.

In [ ]:
# seu código aqui